In [12]:
import pennylane as qml
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn 
import torch.nn.functional as F 

from tqdm import tqdm

# Hamiltonians

In [13]:
def H_mixer(n: int,
            h: np.ndarray|None=None,
            ) -> qml.Hamiltonian:
    """
    n: number of qubits in the system.
    h: external field strength.
    """
    if not h:
        h = np.ones(n)
    obs = [qml.PauliX(i) for i in range(n)]
    return - qml.Hamiltonian(h, obs)

def H_prob(n: int, 
           mu: float,
           h: np.ndarray|None=None,
           coupling: np.ndarray|None=None,
           ) -> qml.Hamiltonian:
    """
    n: number of qubits in the system.
    h: external field strength.
    coupling: couplings between qubits.
    """
    if not h:
        h = np.ones(n)
    if not coupling:
        coupling = np.ones((n, n))
    h_obs = [qml.PauliZ(i) for i in range(n)]
    h_hamiltonian = - mu * qml.Hamiltonian(h, h_obs)
    coupling_obs = []
    for i in range(n):
        for j in range(n):
            coupling_obs.append(qml.PauliZ(i) @ qml.PauliZ(j))
    coupling_hamiltonian = - qml.Hamiltonian(coupling, coupling_obs)
    return h_hamiltonian + coupling_hamiltonian

def H_a(H_mixer: qml.Hamiltonian,
        H_prob: qml.Hamiltonian,
        schedule,
        time: float,
        ) -> qml.Hamiltonian:
    return (1 - schedule(time)) * H_prob + schedule(time) * H_mixer

# The circuit

In [ ]:
def state(n: int, 
          trotter_steps: int,
          H_mixer: qml.Hamiltonian,
          H_prob: qml.Hamiltonian,
          q_device_name: str="default.qubit",
          ):
    
    # Devices
    qdev = qml.device(q_device_name)
    cdev = "cuda" if torch.cuda.is_available() else "cpu"

    @qml.qnode(qdev)
    def circuit():

        # Defining the parameters
        gamma = nn.Parameter(torch.rand(p), requires_grad=True)
        beta = nn.Parameter(torch.rand(p), requires_grad=True)

        # Starting from equal superposition
        for i in range(n):
            qml.Hadamard(i)

        # Parametrized quantum simulation algorithm (the bridge between annealing and digitalized circuit)
        for p in range(trotter_steps):
            qml.evolve(H_prob(wires=range(n)), coeff=gamma[p])
            qml.evolve(H_mixer(wires=range(n)), coeff=beta[p])

        
        